# Notebook 02 — Inspect the Prebuilt Task Set

## Why this notebook exists

The eval's quality is bounded by the quality of its **task set**. Sloppy tasks → sloppy signal, no matter how good the agents under test are.

To keep this workshop reproducible *and* unbiased, we ship a hand-curated task set rather than asking an LLM to invent one. The tasks are grounded in real files in [`aws-samples/sample-agentic-platform`](https://github.com/aws-samples/sample-agentic-platform) at a **pinned SHA**, and were authored by reading the repo by hand. This avoids two problems:

1. **Author bias**: if Claude writes the tasks *and* gets graded on them, you're measuring how well it solves problems in the shape it likes to write — not real-world performance.
2. **Reproducibility drift**: when the upstream repo evolves, line numbers and file paths shift. Pinning the SHA freezes the substrate so your eval results are comparable across runs.

This notebook walks you through:

- **Step 1**: Clone the target repo at the pinned SHA.
- **Step 2**: Validate the prebuilt `scaffolding/tasks/tasks.yaml` against the schema.
- **Step 3**: Inspect one task in detail — see how an issue description, ground-truth scope, and Q&A pairs fit together.
- **Step 4**: Read the schema documentation, so when you adapt this workshop to your own repo, you can author your own tasks.

You will **not** be writing or modifying tasks here. The deliverable is understanding.

## What's in the prebuilt set

Nine tasks across three categories:

| category | count | what it tests |
|---|---|---|
| **Normal** (`is_trap=false, nav_only=false`) | 5 | The core autonomous-eval signal. Easy → hard targeted edits with known correct fixes. |
| **Trap** (`is_trap=true`) | 2 | Honesty. The issue describes a bug that doesn't exist in the code — the agent should investigate and refuse, not fabricate a fix. |
| **Nav-only** (`nav_only=true`) | 2 | Pair-programmer skill. The deliverable is answers, not a diff. Tests retrieval + grounded explanation. |

## Step 1 — Clone the target repo at the pinned SHA

The pinned SHA lives at the top of `scaffolding/tasks/tasks.yaml`. We read it from there and check out exactly that commit. If you ever need to refresh the task set against a newer repo SHA, bump it in the YAML and re-verify line numbers manually.

In [ ]:
import subprocess
from pathlib import Path
import yaml

WORKSHOP_DIR = Path.cwd().resolve()
TASKS_FILE = WORKSHOP_DIR / 'scaffolding' / 'tasks' / 'tasks.yaml'

tasks_doc = yaml.safe_load(TASKS_FILE.read_text())
REPO_URL = tasks_doc['repo']['url']
PINNED_SHA = tasks_doc['repo']['pinned_sha']

CLONE_PATH = Path('/tmp/coding-eval-target/sample-agentic-platform')
CLONE_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CLONE_PATH.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CLONE_PATH)], check=True)

# Make sure the pinned SHA is available locally, then check it out.
subprocess.run(['git', 'fetch', 'origin', PINNED_SHA], cwd=CLONE_PATH, check=True)
subprocess.run(['git', 'checkout', PINNED_SHA], cwd=CLONE_PATH, check=True)

head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=CLONE_PATH,
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_SHA, f'Checkout failed: {head} != {PINNED_SHA}'

print(f'Repo:       {CLONE_PATH}')
print(f'Pinned SHA: {PINNED_SHA}')

## Step 2 — Validate the prebuilt task set

The validator does three things:

1. **Schema check**: every task has the required fields (id, difficulty, issue_description, etc.) with the right shapes.
2. **Population check**: at least 2 traps and 2 nav-only tasks across the set.
3. **Path grounding**: every `affected_paths` and `relevant_files` entry points to a real file in the cloned repo at the pinned SHA. Catches stale paths if anyone bumps the SHA without re-verifying.

If this passes, the task set is structurally sound. Quality is a separate question — for that, you read the tasks (next step).

In [ ]:
import sys
sys.path.insert(0, '.')
from validators.tasks import validate_tasks_file

v = validate_tasks_file(TASKS_FILE, repo_root=CLONE_PATH)
print(v.report())
assert v.passed, 'Prebuilt task set failed validation. See errors above.'

## Step 3 — Inspect a task, end-to-end

A good task has three parts that line up:

1. **The issue description** — sounds like something a human teammate would write. Specific files, specific lines, specific ask. No "improve the code" hand-waving.
2. **The relevant_files** — the *short* list of files an agent really needs to touch or read. This is the IR ground-truth used in notebook 06.
3. **The qa_pairs** — questions about the code surrounding this task, with concrete answers (`path:line`). These drive the pair-programmer eval.

Below: T03 (medium difficulty, hardcoded region in `kb_client.py`). Read each panel and notice how they reinforce each other.

In [ ]:
from IPython.display import Markdown, display

tasks = tasks_doc['tasks']
print(f'{len(tasks)} tasks in the prebuilt set.')
print()

# Pick T03 — a "normal" medium task with rich qa_pairs and clear ground truth.
t = next(t for t in tasks if t['id'] == 'T03_hardcoded_region_in_kb_client')

et = t.get('expected_tools', {}) or {}
required = ', '.join(et.get('required') or []) or '(none)'
forbidden = ', '.join(et.get('forbidden') or []) or '(none)'

display(Markdown(f'''### {t['id']} — {t['title']}

| field | value |
|---|---|
| difficulty | {t['difficulty']} |
| skills | {', '.join(t.get('skills', []))} |
| is_trap | {t.get('is_trap', False)} |
| nav_only | {t.get('nav_only', False)} |
| affected_paths | {', '.join(t.get('affected_paths', []))} |
| relevant_files | {', '.join(t.get('relevant_files', []))} |
| required tools | {required} |
| forbidden tools | {forbidden} |

**Issue description** (this is what the agent actually sees):

{t['issue_description']}
'''))

# qa_pairs — the pair-programmer eval substrate.
display(Markdown('**Q&A pairs** (used in notebook 06 to score retrieval + answer correctness):\n'))
for i, qa in enumerate(t.get('qa_pairs', []) or [], start=1):
    display(Markdown(f'''**Q{i}.** {qa['q']}

> **A:** {qa['a']}
>
> *relevant_files:* `{', '.join(qa.get('relevant_files', []))}`
'''))

### Distribution check

A useful task set spans difficulty and category. Below is the shape of the prebuilt set — note the trap and nav-only counts.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'id': t['id'],
     'difficulty': t['difficulty'],
     'is_trap': t.get('is_trap', False),
     'nav_only': t.get('nav_only', False),
     'n_qa_pairs': len(t.get('qa_pairs', []) or []),
     'n_relevant_files': len(t.get('relevant_files', []) or []),
     'required_tools': ', '.join((t.get('expected_tools', {}) or {}).get('required', []) or []) or '-',
     'issue_chars': len(t['issue_description'])}
    for t in tasks
])
df

## Step 4 — Schema reference (read this when adapting to your own repo)

The full schema lives in `scaffolding/task_schema_example.yaml`. The fields that matter most when authoring new tasks:

| field | required | what it means |
|---|---|---|
| `id` | yes | Stable identifier. Convention: `T<NN>_<short_slug>`. |
| `title` | yes | One-line human description. Shown in scorecards. |
| `difficulty` | yes | `easy` / `medium` / `hard`. Used for difficulty-stratified scoring in notebook 07. |
| `skills` | yes | Free-form tags (e.g. `targeted_edit`, `multi_file_refactor`). Useful for slicing the scorecard. |
| `affected_paths` | yes | Files the agent is *expected* to modify. The scope-discipline check fails any diff that edits files outside this set. |
| `relevant_files` | yes | Files an honest investigation has to read, even if the diff doesn't touch them all. IR ground-truth for notebook 06. Keep it tight — overly broad sets dilute precision@k. |
| `issue_description` | yes | What the agent sees as input. Write it like a real Jira/GitHub issue: concrete, file:line citations, scoped. |
| `expected_tools.required` | optional | Tools that must show up in the trace (e.g. `find_callers` for a refactor across call sites). |
| `expected_tools.forbidden` | optional | Tools that, if used, indicate the agent went off the rails (e.g. `web_search` on a closed-source repo). |
| `qa_pairs` | yes | List of `{q, a, relevant_files}`. 2-3 per task. Used in notebook 06. The `a` should cite `path:line`. |
| `is_trap` | optional | If `true`, the issue describes a non-existent bug. Pass = agent investigates and refuses; fail = agent fabricates a fix. **At least 2 per set.** |
| `nav_only` | optional | If `true`, the deliverable is `qa_pair` answers, not a diff. Skipped by the autonomous eval. **At least 2 per set.** |

### Curating tips that are easy to miss

- **`relevant_files` is not the same as `affected_paths`.** A task that edits `kb_client.py` but requires reading `bedrock_kb_mcp_server/server.py` to discover the existing convention should list both in `relevant_files` but only `kb_client.py` in `affected_paths`.
- **Trap tasks need `relevant_files` too.** They list the files an honest investigation would have looked at — those are the IR targets when the agent does its investigation.
- **Don't write traps that are *too* obvious.** "Why does `add(2, 2)` return 5?" is useless. The fictional bug has to be plausible enough that an over-eager agent would fall for it. The traps in this set describe real-sounding bugs ("missing exception logging", "hardcoded region") in files that already do the right thing.

## Next

Move on to **`03 rubrics and gold standard.ipynb`** to see how each task pairs with a per-dimension rubric and a small set of hand-authored "good diff" / "bad diff" examples that calibrate the LLM judge.